In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras
import numpy as np
import DataPrep


2025-02-10 18:31:29.665267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739208689.677331  144172 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739208689.680788  144172 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-10 18:31:29.696359: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Get Datasets

In [3]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 150
lookahead = 5
batch_size = 20

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP", "BNBUSDT_PERP", "LINKUSD_PERP", "TRXUSD_PERP", "XLMUSD_PERP","DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)

I0000 00:00:1739208691.685587  144172 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


## Combine Datasets

In [4]:
import keras
import os

def load_model(name, optimizer):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
    manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)
    manager.restore_or_initialize()

    return model, manager

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, checkpointmanager):
        super().__init__()
        # no idea if we want to or need to super this
        try:
            self.lastEpoch = int(checkpointmanager.latest_checkpoint.split("-")[-1])
        except Exception as e:
            print(e)
            self.lastEpoch = 0

        self.checkpointManager = checkpointmanager
        print("Model was trained for " + str(self.lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        print(f"Epoch {self.lastEpoch} ended")
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.checkpointManager.save(checkpoint_number=epoch)


In [5]:
#zip the different dataset sources
zipped = tf.data.Dataset.zip(datasets=tuple(datasets))
def combineZippedBatches(*zipped):
    batchshape = zipped[0][0]
    # Create first tensors to concat the rest
    data = zipped[0][0]
    label = zipped[0][1]
    for i in range(1, len(zipped)): # iterate over each remaining pair
        data = tf.concat([data, zipped[i][0]], axis=0)
        label = tf.concat([label, zipped[i][1]], axis=0)

    return data, label

# combine batches into one megabatch
batchTogether = zipped.map(combineZippedBatches)

# Prefetch and create labels
training = batchTogether.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
validation = valdataset.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

## Custom Losses and metrics

In [6]:
def mappedloss(y_true, y_pred):
    marginerror = tf.square(y_true[:] - y_pred[:,0:2])
    predictionerror = tf.square(marginerror - tf.square(y_true[:,2:]))
    combined = tf.concat([marginerror, predictionerror], axis=1)
    return combined

def naiveloss(y_true, y_pred):
    error = tf.abs(y_pred[:,0:2] - y_true)
    predictederror = y_pred[:,2:4]

    return tf.concat([error, tf.abs(error -predictederror)], axis=1)

    #return tf.reduce_mean(tf.square(error - predictederror) + predictederror, axis=1) # add on predicted error to incentivise minimization

def minmax_MSE(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true[:] - y_pred[:,0:2]), axis=1)

def expected_MSE(y_true, y_pred):
    # Calculate mse of expected log variance
    return tf.reduce_mean(tf.square(y_pred[:,2:4]), axis=1)

def error_of_error_MSE(y_true, y_pred):
    return tf.abs(minmax_MSE(y_true, y_pred) - expected_MSE(y_true, y_pred))

def expected_error_variance(y_true, y_pred):
    exerrors = tf.reduce_mean(y_pred[:,2:4], axis=0)
    return tf.math.reduce_variance(exerrors)



## Train!

In [7]:
modelName = "dingus3"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )
optimizer = keras.optimizers.Adam(amsgrad=True, clipvalue=0.2)

model, manager = load_model(modelName, optimizer)

model.compile(loss= mappedloss,#[error_of_error_MSE, minmax_MSE],#naiveloss,
                  optimizer=optimizer,
                  # For the metrics, always have AT LEAST these two
                  #metrics=["MeanAbsolutePercentageError", "MeanSquaredError"])
                  # CUSTOM METRIC!:
                  metrics=[expected_error_variance, minmax_MSE, error_of_error_MSE],
              )

model.summary()

history = model.fit(training, epochs=5, verbose=0, validation_data=validation, callbacks=[saveEachEpoch(manager), tensorboard])

/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 50 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 6, 5, 150) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 4500)      │          0 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 6000)      │ 27,006,000 │ flatten_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 6000)      │          0 │ dense_26[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 6000)      │ 36,006,000 │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 6000)      │          0 │ dense_27[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 4000)      │ 24,004,000 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 4000)      │          0 │ dense_28[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 4000)      │ 16,004,000 │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 4000)      │          0 │ dense_29[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 1000)      │  4,001,000 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 5000)      │  5,005,000 │ dense_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ minmaxprediction    │ (None, 2)         │     10,002 │ dense_31[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ errorprediction     │ (None, 2)         │     10,002 │ dense_31[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 4)         │          0 │ minmaxprediction… │
│ (Concatenate)       │                   │            │ errorprediction[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 112,046,004 (427.42 MB)

 Trainable params: 112,046,004 (427.42 MB)

 Non-trainable params: 0 (0.00 B)

Model was trained for 1 epochs before.


ValueError: Dimensions must be equal, but are 2 and 0 for '{{node compile_loss/mappedloss/sub_1}} = Sub[T=DT_FLOAT](compile_loss/mappedloss/Square, compile_loss/mappedloss/Square_1)' with input shapes: [160,2], [160,0].